# Benchmark de ASR en GPU (Tarea 1, Fase 2) + transcripción del corpus

**Antes de correr:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
Corre las celdas **en orden**. Si una falla, copia el mensaje de error completo y envíamelo.
Tiempo estimado: 15–30 min (la primera vez baja ~3 GB de modelos).

In [ ]:
# 1) ¿Hay GPU? (si dice "failed", cambia el tipo de entorno a T4 GPU y reinicia)
!nvidia-smi

In [ ]:
# 2) Clonar TU repo (debe ser público y tener ya mergeado el código del motor).
#    Cambia TU_USUARIO. Si aún no hiciste merge a main, pon BRANCH = "feat/motor-asr".
REPO_URL = "https://github.com/TU_USUARIO/voice-to-record.git"
BRANCH = "main"
!rm -rf repo && git clone --branch $BRANCH $REPO_URL repo
%cd repo

In [ ]:
# 3) Instalar SOLO lo necesario, con las versiones fijadas del repo (no uses requirements.txt completo en Colab)
!pip install -q faster-whisper==1.2.1 ctranslate2==4.8.4 jiwer==4.0.0 pyyaml==6.0.3
import ctranslate2
print("GPUs visibles para ctranslate2:", ctranslate2.get_cuda_device_count())
assert ctranslate2.get_cuda_device_count() > 0, "ctranslate2 no ve la GPU: revisa el paso 1"

In [ ]:
# 4) Subir los 10 audios del SET A (t1_01 ... t1_05, t2_01 ... t2_05). Solo los tuyos.
from google.colab import files
import os, shutil
os.makedirs("data/audio", exist_ok=True)
for nombre in files.upload():
    shutil.move(nombre, f"data/audio/{nombre}")
print(sorted(os.listdir("data/audio")))

In [ ]:
# 5) Duración de cada clip (debe estar entre 120 y 240 s)
!for f in data/audio/*; do echo -n "$f: "; ffprobe -v error -show_entries format=duration -of csv=p=0 "$f"; done

In [ ]:
# 6) BENCHMARK de la Tarea 1 (modelos small y large-v3, GPU/CPU, float16/int8, glosario)
!python scripts/benchmark_asr.py --task t1

In [ ]:
# 7) Resultado en tablas
print(open("results/asr_benchmark_t1.md", encoding="utf-8").read())

In [ ]:
# 8) Transcribir el corpus de la Tarea 2 con la configuración de asr.cache_corpus (large-v3, cuda, float16)
!python scripts/transcribir_corpus.py --task t2 --model large-v3 --device cuda --compute float16

In [ ]:
# 9) Empaquetar y descargar (results/ y data/cache/ se commitean; los audios NO)
!zip -qr salida_colab.zip results data/cache
files.download("salida_colab.zip")

## Después de descargar
1. Descomprime `salida_colab.zip` **dentro de tu repo local** (se mezcla con `results/` y `data/cache/`).
2. `git add results data/cache && git commit -m "feat: benchmark ASR y transcripciones en caché"` y abre un pull request.
3. Cuéntame qué dice la tabla (especialmente la columna *GPU verificada*) y seguimos con la extracción.